# PySpark and SEC EDGAR (Government Financial Data)

## How This Notebook Works

This file is a **Jupyter Notebook** — a document format with the extension `.ipynb` (short for **I**nteractive **Py**thon **N**ote**b**ook). It mixes two types of cells:

| Cell Type | What It Does |
|-----------|-------------|
| **Code (Python)** | Contains runnable Python code. Click the ▶ Run button on the left (or press **Shift + Enter**) to execute it. |
| **Text (Markdown)** | Contains formatted text like this cell — headings, tables, bullet points, etc. It does not run. |

**Important:** Python cells do **not** run automatically when you open the notebook. Each one must be run manually.

**Run cells in order.** Earlier cells often declare variables, import libraries, or configure settings that later cells depend on. Skipping a cell or running them out of order can cause a `NameError` ("variable not defined") in a later cell.

## What Are Spark, PySpark, and Databricks?

### Apache Spark
**Apache Spark** is an open-source engine for processing large amounts of data very quickly. Instead of running on a single computer, Spark splits data across many machines (a "cluster") and processes all the pieces at the same time. This makes it practical to analyze datasets too large to fit on one computer — think hundreds of millions of rows in seconds.

### PySpark
**PySpark** is the Python interface for Spark. Because Spark was originally written in a language called Scala, PySpark exists so Python developers can write Spark code without learning Scala. Every PySpark command is translated into Spark instructions behind the scenes — so I get Python's readability while Spark handles the heavy lifting.

### Databricks
**Databricks** is a managed cloud platform built on top of Spark. Think of it as "Spark with the setup already done." Instead of manually installing Spark, configuring machines, and managing clusters, Databricks provides a ready-to-use environment with:
- A notebook interface (like this one) that runs directly on a Spark cluster
- Automatic cluster management (start, scale, and shut down)
- Built-in storage, scheduling, and collaboration tools

In short: **Spark** is the engine, **PySpark** is the Python steering wheel, and **Databricks** is the car that comes pre-assembled and ready to drive.

## How to Run This Notebook

Choose the option that fits your setup:

| Option | Best For | Steps |
|--------|----------|-------|
| **Databricks** (cloud) | Production-like Spark, portfolio demos | 1. Sign up at databricks.com -> 2. Create a cluster -> 3. Import this `.ipynb` -> 4. Attach notebook to cluster -> 5. Run All |
| **Local (VS Code / Jupyter)** | Offline development, quick edits | 1. `pip install pyspark` -> 2. Open this file in VS Code or Jupyter -> 3. Run cells top to bottom |
| **Google Colab** (free cloud) | No install, browser-only | 1. Upload this file to colab.research.google.com -> 2. Add `!pip install pyspark` as the very first cell -> 3. Run all cells |

**Recommendation:** Use **Databricks** for the most faithful experience — it mirrors how this notebook would run in a real data engineering environment. Use **Local** for fast iteration during development.

## Step 1: Set Initial Settings

In [ ]:
import os
from dotenv import load_dotenv  # Reads key=value pairs from .env and injects them into os.environ

# Load .env so machine-specific config (e.g. JAVA_HOME) is available before anything else runs.
# load_dotenv() silently does nothing if .env is missing — no crash, no side effects.
# override=False means existing environment variables (e.g. set in .zshrc) take priority over .env values.
load_dotenv(override=False)

# Set JAVA_HOME so PySpark can find Java when running outside Databricks (e.g. VS Code, Jupyter).
# Jupyter kernels don't inherit shell environment variables, so this must be set in code before
# SparkSession is created — otherwise the Java gateway fails to start and Spark never launches.
# The value comes from .env (gitignored), not hardcoded here, so the path stays off GitHub.
java_home = os.environ.get("JAVA_HOME")
if not java_home:
    raise EnvironmentError(
        "JAVA_HOME is not set. Copy .env.example to .env and fill in your Java path.\n"
        "To find it on macOS, run: /usr/libexec/java_home"
    )
os.environ["JAVA_HOME"] = java_home

# Import PySpark SQL functions for data transformations (filtering, aggregating, etc.)
from pyspark.sql import SparkSession, DataFrame, Column, functions as F  # DataFrame and Column are imported here so they can be used as type hints in all functions throughout this notebook
from pyspark.sql.types import StructType, StructField, StringType  # Explicit imports for schema definition used in Step 2
from pyspark.sql.window import Window  # Import for ranking and time-series operations
from typing import Optional, Any  # Optional marks return types that can be None; Any types dict values whose structure isn't fixed at write-time

# Import Python libraries for API calls and parallel processing
import requests  # Make HTTP requests to SEC EDGAR API
from concurrent.futures import ThreadPoolExecutor  # Run multiple API calls in parallel
import time  # Add delays to respect SEC rate limits
from datetime import datetime, date  # Handle date/time operations; date is used as a return type hint for collected Spark DateType values

# ── SPARKSESSION ──────────────────────────────────────────────────────────────
# Source: spark.apache.org/docs/latest/sql-getting-started.html -> "Starting Point: SparkSession"
# On Databricks this line runs automatically behind the scenes; outside it (VS Code, Jupyter), you need to write it into your code yourself.
#
#   SparkSession.builder            — opens a config builder
#   .appName("SEC EDGAR Analytics") — labels this job in Spark's UI and logs
#   .getOrCreate()                  — starts Spark if not running; reuses an existing session if it is
#
# ─────────────────────────────────────────────────────────────────────────────

# Spark has to launch a Java Virtual Machine (JVM) before it can do anything.
# That process takes 30-60 seconds on first run — the cell will look frozen during this time, which is normal.
print("Starting Spark (JVM startup takes 30-60 seconds on first run — please wait)...")

spark: SparkSession = (
    SparkSession.builder
    .appName("SEC EDGAR Analytics")  # Identifies this job in Spark's UI and logs
#   .config("spark.some.config.option", "some-value")
    .getOrCreate()                   # Returns existing session on Databricks; creates new one locally
)

# ========================================
# WHY EXPLICIT CONFIGURATION MATTERS
# ========================================
# Production code principle: Always explicitly set critical configurations, even if they're defaults
# Reasons: (1) Makes code portable across different Spark versions and cluster setups
#          (2) Self-documents your intentions for anyone reading the code
#          (3) Protects against cluster-level overrides that might disable features
# ========================================

# ========================================
# ADAPTIVE QUERY EXECUTION (AQE)
# ========================================
# What it is: Feature that watches your query run and adjusts the execution plan in real-time
# How it works: After each stage completes, Spark collects actual statistics (row counts, data sizes)
#               and compares them to initial estimates. If there's a mismatch, it adjusts:
#               - Merges many small partitions into fewer larger ones (less overhead)
#               - Switches join strategies when one table is smaller than expected
#               - Splits oversized partitions to prevent one worker from being overwhelmed
# Why specify it: Even though it's default in Spark 3.2+, explicit setting ensures it works on
#                 older Spark versions (2.4, 3.0) and overrides any cluster-level disabling
# Why it matters here: SEC filing data is highly skewed (some companies file 100x more than others)
#                      AQE automatically handles this without manual partition tuning
# ========================================

# AQE-specific settings (lines below configure Adaptive Query Execution features)
spark.conf.set("spark.sql.adaptive.enabled", "true")  # Master switch: turns on AQE's real-time optimization engine
spark.conf.set("spark.sql.adaptive.coalescePartitions.enabled", "true")  # Partition merging: combines 100 tiny data chunks into 5 normal ones (reduces task overhead)
spark.conf.set("spark.sql.adaptive.skewJoin.enabled", "true")  # Skew handling: splits Apple's 10K filings across multiple workers instead of overwhelming one (prevents bottlenecks)

# Why enable these specific AQE features?
# 1. Master switch (adaptive.enabled): Without this, the other two settings do nothing - it's the on/off button
# 2. Partition merging (coalescePartitions): SEC data varies wildly - one company might have 5 filings, another 500
#    Without merging, Spark creates too many tiny tasks for small companies, wasting time on coordination
# 3. Skew handling (skewJoin): When joining company info with filings, megacorps like Berkshire Hathaway have 1000x
#    more filings than a small startup. Without skew handling, one worker gets stuck processing Berkshire while
#    others sit idle. This feature splits the work evenly across all workers.

# General Spark settings (not AQE-specific)
spark.conf.set("spark.sql.shuffle.partitions", "200")  # Default partitions for distributed operations (groupBy, joins)

# Display settings for cleaner notebook output
spark.conf.set("spark.sql.repl.eagerEval.enabled", "true")  # Show DataFrames without calling .show()
spark.conf.set("spark.sql.repl.eagerEval.maxNumRows", "10")  # Limit automatic preview to 10 rows

print("Setup complete! Spark configured with Adaptive Query Execution enabled.")
print(f"Spark version: {spark.version}")
print(f"AQE Status: {spark.conf.get('spark.sql.adaptive.enabled')}")

## Step 2: Ingest SEC EDGAR Data

In [ ]:
# ════════════════════════════════════════════════════════════════
# STEP 2: INGEST SEC EDGAR DATA
# ════════════════════════════════════════════════════════════════
# Goal: Download recent filing records for a chosen set of
# companies from the SEC's public API and load them into Spark.
#
# This cell is split into four parts:
#   1. Configuration  — constants and company identifiers
#   2. Schema         — the table blueprint Spark will use
#   3. Functions      — reusable building blocks for fetching data
#   4. Run            — calls the functions and shows the result
# ════════════════════════════════════════════════════════════════

# ════════════════════════════════════════════════════════════════
# WHY I USE THREADS HERE — NOT PYSPARK'S DISTRIBUTED CLUSTER
# ════════════════════════════════════════════════════════════════
# The work in this step is I/O-bound: I'm waiting for the SEC
# server to respond to HTTP requests, not crunching large datasets.
#
# PySpark's multi-machine distribution shines when the bottleneck
# is compute or data volume — e.g. joining billions of rows across
# multiple servers.  Spinning up that infrastructure to fetch a few
# hundred API responses would add significant startup time and cost
# for no real gain, and would actually slow the job down.
#
# Python threads (ThreadPoolExecutor) are the right tool here:
#   - They overlap the network-waiting time across many requests
#     simultaneously on the same machine, without any Spark overhead.
#   - A single machine with MAX_WORKERS threads is also far easier
#     to rate-limit correctly — the SEC caps requests at 10/s, and
#     one machine with RATE_LIMIT_DELAY = 0.12 s enforces that cap
#     reliably.  Spreading requests across many Spark executors on
#     different machines would make coordinating that shared rate
#     limit much more complex and error-prone.
#
# Best practice summary:
#   - Use threads  -> lightweight I/O fetch (this step)
#   - Use PySpark  -> large-scale data transformation (Steps 3+)
# ════════════════════════════════════════════════════════════════

import json  # Needed to catch JSON parsing errors when the SEC API returns malformed responses
import html  # Provides html.escape() to safely neutralize HTML characters before logging untrusted content


# ── PART 1: CONFIGURATION ────────────────────────────────────────────────────
# All important constant values (URLs, limits, identifiers) are here
# so they're easy to find and change without digging through the rest of the code.

URL_BASE_EDGAR: str    = "https://data.sec.gov/submissions"       # Root URL for the SEC filing API
URL_ALL_COMPANIES: str = "https://www.sec.gov/files/company_tickers.json"  # SEC list of every registered company

# The SEC EDGAR API requires every request to identify who is making it.
# Without this header the server returns a 403 "Forbidden" error and blocks you.
HEADERS: dict[str, str] = {"User-Agent": "financial-portfolio DaveDevPortfolio@gmail.com"}

MAX_WORKERS: int         = 5     # How many companies to fetch at the same time (explained in Part 3)
RATE_LIMIT_DELAY: float  = 0.12  # Seconds to pause before each request — keeps me under SEC's 10 req/s cap

# Single source of truth for all request timeouts — change this one value to adjust every API call in this notebook
REQUEST_TIMEOUT_SECONDS: int = 10

# ── COST CONTROL GATE ────────────────────────────────────────────────────────
# Databricks charges based on cluster compute time.  Querying all ~10,000
# SEC-registered companies at once would multiply that cost dramatically.
# This cap limits any bulk mode to 500 companies at most, striking a balance
# between broad coverage and keeping the job affordable.
MAX_COMPANIES_LIMIT: int = 500

# All valid values for QUERY_MODE — used to catch typos before any network calls happen
VALID_QUERY_MODES: set[str] = {"SPECIFIC", "FIRST_500", "ALL"}


# ════════════════════════════════════════════════════════════════
# CHOOSE YOUR QUERY MODE
# ════════════════════════════════════════════════════════════════
# Set QUERY_MODE to one of three options:
#
#   "SPECIFIC"   ->  Query only the hand-picked CIKs in CIKS_TO_QUERY below.
#                    Best for targeted analysis of a known list of companies.
#
#   "FIRST_500"  ->  Automatically pull the first 500 companies from SEC EDGAR,
#                    further capped by MAX_COMPANIES_LIMIT (so if the cap is 100,
#                    only 100 companies are fetched, not 500).
#                    The SEC orders its list by CIK number (roughly chronological
#                    registration order), so this gives the earliest-registered
#                    companies — not the largest or most prominent.
#                    Good for broad analysis without manual curation.
#
#   "ALL"        ->  Pull every company registered with the SEC (~10,000+),
#                    then cap the result at MAX_COMPANIES_LIMIT to
#                    control Databricks compute costs.
# ════════════════════════════════════════════════════════════════

QUERY_MODE: str = "SPECIFIC"  # <-- Change this value to switch modes


# ── OPTION 1: SPECIFIC COMPANIES ─────────────────────────────────────────────
# Add or remove entries to query exactly the companies you care about.
#
# HOW TO FIND A COMPANY'S CIK NUMBER:
#   1. Go to https://www.sec.gov/cgi-bin/browse-edgar?action=getcompany
#   2. Type the company name (e.g. "Tesla") in the search box and hit Enter.
#   3. The CIK appears as a 10-digit number next to the company name in the results.
#   4. You can paste the raw number — leading zeros are added automatically below.

CIKS_TO_QUERY: list[str] = [
    "0000320193",  # Apple
    "0000789019",  # Microsoft
    "0001018724",  # Amazon
    "0001652044",  # Alphabet (Google)
    "0001326801",  # Meta
    "0001045810",  # NVIDIA
    "0000051143",  # IBM
    "0000078814",  # JPMorgan Chase
]


# ── PART 2: SCHEMA ───────────────────────────────────────────────────────────
# A schema is a blueprint that tells Spark the exact column names and data types
# before it reads any data.  Without one, Spark guesses — and sometimes guesses
# wrong (e.g. treating a date as plain text, or an ID number as an integer).
#
# StructType([...])               -> the full blueprint (all columns together)
# StructField(name, type, nullable)
#   name     -> column label in the resulting table
#   type     -> StringType() means "store as plain text"
#   nullable -> True means a missing/null value is allowed in this column

FILINGS_SCHEMA: StructType = StructType([
    StructField("cik",             StringType(), True),  # Company ID — kept as text because math is never done on it
    StructField("form",            StringType(), True),  # Filing type, e.g. "10-K" (annual report) or "8-K" (major event)
    StructField("filingDate",      StringType(), True),  # Date stored as text for now; will be cast to a real date in Step 3
    StructField("accessionNumber", StringType(), True),  # SEC's unique tracking ID for each individual filing document
])


# ── PART 3: FUNCTIONS ────────────────────────────────────────────────────────

def resolve_ciks(mode: str, specific_ciks: list[str]) -> list[str]:
    """
    Returns the final list of CIK strings to query, based on the chosen mode.

    - "SPECIFIC"  -> normalizes and returns the hand-picked list.
    - "FIRST_500" -> fetches all SEC-registered companies and returns the first 500,
                     capped at MAX_COMPANIES_LIMIT (whichever is smaller).
    - "ALL"       -> fetches all SEC-registered companies and returns up to MAX_COMPANIES_LIMIT.

    The SEC's company_tickers.json endpoint returns a dict keyed by sequential
    integers ("0", "1", "2" ...), each containing a company's CIK, ticker, and name.
    CIKs in that file are plain integers so they must be zero-padded to 10 digits
    to match the format the filings API expects.

    Raises ValueError if mode is not one of the three valid options.
    Raises RuntimeError if the SEC company list cannot be fetched or parsed.
    """
    # Catch typos in QUERY_MODE early — before any network calls are made
    if mode not in VALID_QUERY_MODES:
        raise ValueError(
            f"QUERY_MODE '{mode}' is not valid. "
            f"Choose one of: {', '.join(sorted(VALID_QUERY_MODES))}"
        )

    # .zfill(10) pads the left side of the CIK with zeros until it is exactly 10 characters long
    # (e.g. "320193" becomes "0000320193") — the SEC API requires this exact format for every mode,
    # so normalizing here means the user never has to worry about adding leading zeros manually
    if mode == "SPECIFIC":
        # Guard against an empty list — nothing to fetch if no CIKs were provided
        if not specific_ciks:
            raise ValueError("CIKS_TO_QUERY is empty. Add at least one CIK before running in SPECIFIC mode.")
        return [str(cik).zfill(10) for cik in specific_ciks]

    # Fetch the full company list from SEC (used for both FIRST_500 and ALL)
    try:
        response = requests.get(URL_ALL_COMPANIES, headers=HEADERS, timeout=REQUEST_TIMEOUT_SECONDS)
        response.raise_for_status()  # Raises HTTPError for 4xx/5xx responses
        companies = response.json()  # Returns a dict: {"0": {cik_str, ticker, title}, "1": {...}, ...}
    except requests.exceptions.Timeout:
        # The SEC server took longer than REQUEST_TIMEOUT_SECONDS — likely a temporary slowdown on their end
        raise RuntimeError(
            f"Request to {URL_ALL_COMPANIES} timed out after {REQUEST_TIMEOUT_SECONDS} seconds. "
            "The SEC server may be slow — wait a moment and try again, or increase REQUEST_TIMEOUT_SECONDS."
        )
    except requests.exceptions.ConnectionError:
        # No network path to the SEC server — check internet connectivity
        raise RuntimeError(
            f"Could not connect to {URL_ALL_COMPANIES}. "
            "Check your internet connection or whether the SEC EDGAR service is reachable."
        )
    except requests.exceptions.HTTPError as http_err:
        # The server responded but returned an error status (e.g. 403 Forbidden, 500 Server Error)
        raise RuntimeError(
            f"SEC server returned an error fetching the company list: {http_err}. "
            "Verify that the User-Agent header in HEADERS is set correctly."
        )
    except (json.JSONDecodeError, ValueError):
        # The server replied but the body wasn't valid JSON — rare, usually a temporary SEC issue
        raise RuntimeError(
            "The SEC company list response could not be parsed as JSON. "
            "The SEC server may have returned an error page — try again in a few minutes."
        )

    # Pull out every CIK and zero-pad it to 10 digits so it matches the filings API format
    try:
        all_ciks: list[str] = [str(v["cik_str"]).zfill(10) for v in companies.values()]
    except (KeyError, AttributeError):
        # The JSON structure was valid but missing the expected "cik_str" field — API may have changed
        raise RuntimeError(
            "The SEC company list JSON was in an unexpected format. "
            "The SEC may have updated their API structure — check URL_ALL_COMPANIES."
        )

    limited: list[str]
    if mode == "FIRST_500":
        # min() ensures MAX_COMPANIES_LIMIT is always respected, even if it's set below 500
        limited = all_ciks[:min(500, MAX_COMPANIES_LIMIT)]
    else:  # "ALL" — take everything but cap at MAX_COMPANIES_LIMIT to control Databricks costs
        limited = all_ciks[:MAX_COMPANIES_LIMIT]

    # Logs a summary so I can confirm which mode ran and how many companies were selected vs. the total available — e.g. "[info] Mode 'ALL': 10,842 companies found, using 500 (cap = 500)"
    print(f"[info] Mode '{mode}': {len(all_ciks):,} companies found, using {len(limited):,} (cap = {MAX_COMPANIES_LIMIT})")
    return limited


def fetch_filings(cik: str) -> list[dict[str, Any]]:
    """
    Downloads filing records for a single company from the SEC API.

    The SEC EDGAR API returns data in "column format" — one list per field — instead of
    the more common "row format" where each record is a complete object.
    This function converts the SEC's column format into a list of row dictionaries
    that Spark can easily load into a table.

    Returns a list of dicts (one per filing), or an empty list if the request fails.
    Errors are logged but not re-raised so one bad CIK doesn't stop the whole job.
    """
    time.sleep(RATE_LIMIT_DELAY)  # Pause before every request so I never exceed SEC's rate limit

    url: str = f"{URL_BASE_EDGAR}/CIK{cik}.json"

    try:
        response = requests.get(url, headers=HEADERS, timeout=REQUEST_TIMEOUT_SECONDS)
        response.raise_for_status()  # If the server returned an error (4xx/5xx), stop and raise an exception

    except requests.exceptions.Timeout:
        # The SEC server took longer than REQUEST_TIMEOUT_SECONDS — log and skip rather than hanging the whole job
        print(f"[warn] CIK {cik}: request timed out after {REQUEST_TIMEOUT_SECONDS}s — skipping. (Increase REQUEST_TIMEOUT_SECONDS if this happens often.)")
        return []

    except requests.exceptions.ConnectionError:
        # A network-level failure (DNS lookup failed, connection refused, etc.)
        print(f"[warn] CIK {cik}: network connection error — check internet connectivity.")
        return []

    except requests.exceptions.HTTPError as http_err:
        status = http_err.response.status_code if http_err.response is not None else "unknown"
        if status == 404:
            # A 404 means the CIK has no filing page — common for recently deregistered companies
            print(f"[warn] CIK {cik}: not found on SEC EDGAR (HTTP 404) — skipping.")
        elif status == 429:
            # A 429 means the SEC rate limiter was hit — increase RATE_LIMIT_DELAY if this happens often
            print(f"[warn] CIK {cik}: rate limit exceeded (HTTP 429) — consider increasing RATE_LIMIT_DELAY.")
        else:
            print(f"[warn] CIK {cik}: HTTP {status} error — {http_err}")
        return []

    # Parse the JSON response — if parsing fails, log a sanitized snippet of the raw response so it's
    # clear whether the SEC returned an error page, a maintenance message, or something unexpected.
    try:
        data: dict[str, Any] = response.json()
    except (json.JSONDecodeError, ValueError):
        # html.escape() is Python's built-in sanitizer — it converts &, <, > into safe display text
        # (&amp;, &lt;, &gt;) so no HTML or script tags from an untrusted server can ever be rendered.
        # It handles the & first internally, avoiding double-encoding — no manual ordering needed.
        raw_preview = (
            html.escape(response.text[:300].strip())  # Cap length, then neutralize all HTML in one trusted call
            .replace("\n", " ")   # Collapse newlines for clean single-line log output
            .replace("\r", "")    # Strip carriage returns (Windows-style line endings)
        )
        print(
            f"[warn] CIK {cik}: response was not valid JSON "
            f"(HTTP {response.status_code}). "
            f"Raw response: {raw_preview!r}"
        )
        return []

    # Navigate to the "recent filings" section of the JSON response.
    # .get() with a default of {} means "if this key is missing, return an empty dict" — safe fallback.
    recent: dict[str, Any] = data.get("filings", {}).get("recent", {})

    # ── WHY THE SEC DATA LOOKS "STRANGE" ─────────────────────────────────
    # Most APIs return rows:  [{"form": "10-K", "date": "2024-10-30"}, ...]
    # The SEC returns columns: {"form": ["10-K", "8-K", ...], "date": ["2024-10-30", ...]}
    #
    # This is the SEC EDGAR API's design — it compresses data for faster transfer.
    # I must reassemble it into rows manually.
    #
    # Step A — extract each column (safe if a field is missing):
    forms: list[str]      = recent.get("form",            [])  # List of filing types
    dates: list[str]      = recent.get("filingDate",      [])  # List of corresponding dates
    accessions: list[str] = recent.get("accessionNumber", [])  # List of corresponding SEC document IDs
    #
    # Step B — zip() stitches the three lists together side-by-side:
    #   forms[0] + dates[0] + accessions[0]  ->  row 1
    #   forms[1] + dates[1] + accessions[1]  ->  row 2  ...
    # ─────────────────────────────────────────────────────────────────────

    # Guard against mismatched column lengths — because the SEC sends data as parallel arrays,
    # forms[0] must pair with dates[0] and accessions[0] for every index.
    # If one list is shorter than the others, zip() silently truncates to the shortest,
    # which would pair the wrong filing type with the wrong date and silently corrupt the data.
    if not (len(forms) == len(dates) == len(accessions)):
        print(
            f"[warn] CIK {cik}: column lengths don't match "
            f"(forms={len(forms)}, dates={len(dates)}, accessions={len(accessions)}) — skipping."
        )
        return []

    # filing_date is used instead of date to avoid shadowing the datetime.date class imported in Step 1
    return [
        {"cik": cik, "form": form, "filingDate": filing_date, "accessionNumber": acc}
        for form, filing_date, acc in zip(forms, dates, accessions)
    ]


def fetch_all_filings(ciks: list[str], max_workers: int = MAX_WORKERS) -> list[dict[str, Any]]:
    """
    Fetches filings for every company in the list, running multiple requests
    at the same time to save time.

    Uses a ThreadPoolExecutor — a tool that runs several tasks simultaneously on
    the same machine using "threads" (independent workers sharing the same CPU).
    This is different from PySpark, which distributes work across multiple physical
    machines. Here, threads overlap the waiting time while the SEC server responds,
    instead of waiting for each company to finish before starting the next.

    Returns a single flat list of row dicts covering all companies.
    Raises ValueError if the CIK list is empty.
    """
    # Catch an empty list early — there's nothing to fetch and the result would always be empty
    if not ciks:
        raise ValueError("The CIK list passed to fetch_all_filings is empty — nothing to fetch.")

    # Creates up to max_workers threads; remaining CIKs queue up and are picked up as threads finish
    with ThreadPoolExecutor(max_workers=max_workers) as pool:
        results: list[list[dict[str, Any]]] = list(pool.map(fetch_filings, ciks))  # Each call returns one company's list of filings

    # ── FLATTENING: turning a list-of-lists into one single list ─────────────
    # results looks like: [ [Apple row1, Apple row2], [Microsoft row1], ... ]
    # Spark needs:        [ Apple row1, Apple row2, Microsoft row1, ... ]
    # The double loop below collapses the nested structure into one flat list.
    # ─────────────────────────────────────────────────────────────────────────
    flat_rows: list[dict[str, Any]] = [row for company in results for row in company]

    # Warn if every single company came back empty — likely a network or configuration issue
    if not flat_rows:
        print(
            "[warn] fetch_all_filings: no filing rows were returned for any company. "
            "Check your internet connection, HEADERS, and whether the SEC API is reachable."
        )

    return flat_rows


def build_filings_dataframe(rows: list[dict[str, Any]]) -> Optional[DataFrame]:
    """
    Loads a flat list of filing-row dicts into a Spark DataFrame using the predefined schema.

    Returns None if rows is empty so the caller can handle the no-data case gracefully
    rather than creating an empty DataFrame that might cause confusing downstream errors.
    """
    # Return None early so the RUN section can print a clear message instead of an empty table
    if not rows:
        return None

    try:
        return spark.createDataFrame(rows, schema=FILINGS_SCHEMA)
    except Exception as spark_err:
        # Spark rejected the data because one or more rows have a field whose Python type doesn't
        # match what FILINGS_SCHEMA declares (e.g. an integer where a string is expected).
        # Scan every row to pinpoint exactly which rows are wrong and which column is the culprit,
        # so you can fix the issue in bulk rather than hunting for it one row at a time.
        schema_fields: dict[str, Any] = {field.name: field.dataType for field in FILINGS_SCHEMA.fields}
        bad_rows: list[tuple[int, str, str, Any]] = []
        for i, row in enumerate(rows):
            for col, expected_type in schema_fields.items():
                val = row.get(col)
                # StringType expects a Python str; flag anything else (int, float, list, etc.)
                if isinstance(expected_type, StringType) and val is not None and not isinstance(val, str):
                    bad_rows.append((i, col, type(val).__name__, val))

        if bad_rows:
            summary: str = "\n".join(
                f"  Row {idx}: column '{col}' has type '{actual_type}' (value: {repr(val)}) — expected str"
                for idx, col, actual_type, val in bad_rows[:20]  # Cap at 20 lines to avoid flooding output
            )
            count_msg: str = (
                f"{len(bad_rows)} row(s) with type mismatches found"
                + (" (showing first 20)" if len(bad_rows) > 20 else "")
            )
            raise RuntimeError(
                f"Spark could not create the filings DataFrame: {spark_err}.\n"
                f"{count_msg}:\n{summary}\n"
                "Tip: if many rows share the same bad column (e.g. all 29 'filingDate' rows show "
                "type 'int'), wrapping that one field in str() inside fetch_filings will fix all "
                "of them at once — no need to edit rows individually."
            ) from spark_err
        else:
            # The type-mismatch scan above found no obvious culprit, so the failure has a different cause.
            # Possible reasons: a field name in the row dicts doesn't match FILINGS_SCHEMA exactly
            # (e.g. "filing_date" vs "filingDate"), a non-nullable column received a None value,
            # or an unexpected Spark-level error unrelated to the data shape.
            raise RuntimeError(
                f"Spark could not create the filings DataFrame: {spark_err}. "
                "No column-type mismatches were detected — check that every key in the row dicts "
                "matches a field name in FILINGS_SCHEMA exactly, that no non-nullable column "
                "contains None, and that the Spark session is still active."
            ) from spark_err


# ── PART 4: RUN ──────────────────────────────────────────────────────────────
# Resolve which CIKs to query based on QUERY_MODE, then fetch and load into Spark.
# All the real logic lives in the functions above — this section stays intentionally short.

active_ciks: list[str]           = resolve_ciks(QUERY_MODE, CIKS_TO_QUERY)      # Determine the final CIK list based on chosen mode
rows: list[dict[str, Any]]       = fetch_all_filings(active_ciks)               # Download filings for all companies in parallel
filings_raw: Optional[DataFrame] = build_filings_dataframe(rows)                # Load into Spark with a guaranteed schema

# Stop early with a clear message if no data came back — avoids a confusing NullPointerError later
if filings_raw is None:
    print("[error] No filing data was loaded into Spark. Check the warnings above for details.")
else:
    print(f"Fetched {filings_raw.count()} filings across {len(active_ciks)} companies  (mode: {QUERY_MODE})")
    filings_raw  # Display a preview (enabled by eagerEval in Step 1)

## Step 3: Clean and Type-Cast Filings Data

In [ ]:
# ════════════════════════════════════════════════════════════════
# STEP 3: CLEAN AND TYPE-CAST FILINGS DATA
# ════════════════════════════════════════════════════════════════
# Goal: Transform filings_raw (all strings) into a properly-typed
# Spark DataFrame where dates are real dates, derived columns are
# added for easier analysis, and unreadable rows are removed.
#
# filings_raw was intentionally kept as all-strings in Step 2 to
# avoid dropping any rows during ingestion. Step 3 is where types
# are enforced and data quality is validated.
# ════════════════════════════════════════════════════════════════

# ── PART 1: CONFIGURATION ────────────────────────────────────────────────────
# Maps raw SEC form codes to plain-English labels for easier reading and filtering.
# Stored here (not buried in a function) so it's easy to extend without touching logic.
FORM_CATEGORIES: dict[str, str] = {
    "10-K":    "Annual Report",
    "10-Q":    "Quarterly Report",
    "8-K":     "Current Event",
    "DEF 14A": "Proxy Statement",
    "S-1":     "IPO Filing",
    "4":       "Insider Trade",
}


# ── PART 2: FUNCTIONS ────────────────────────────────────────────────────────

def clean_filings(df: Optional[DataFrame]) -> Optional[DataFrame]:
    """
    Takes the raw all-string filings DataFrame and returns a cleaned version with:
    - filingDate cast from text ("2024-10-30") to a real Spark DateType
    - filing_year extracted as an integer for fast year-based filtering
    - form_category mapped from raw SEC codes to plain-English labels
    - Rows with unparseable dates removed (logged so nothing is silently lost)

    Accepts Optional[DataFrame] so the None propagated from Step 2 is handled here
    rather than crashing at the call site. Returns None when df is None.
    """
    if df is None:
        return None  # Propagate the None from Step 2 so the caller can handle it cleanly

    # ── WHAT IS "F"? ─────────────────────────────────────────────────────────
    # "F" is a short alias for pyspark.sql.functions — the standard library of
    # built-in Spark tools for transforming data (dates, strings, math, maps, etc.).
    # It was imported at the top of Step 1 as:  from pyspark.sql import functions as F
    #
    # Using "F" (rather than typing "pyspark.sql.functions" every time) is the
    # industry-standard convention in PySpark — the same way Python data engineers
    # use "np" for NumPy or "pd" for pandas.  Renaming it to something longer
    # would go against that convention and make the code harder to read for anyone
    # already familiar with PySpark.  Keeping it as "F" is the right call.
    # ─────────────────────────────────────────────────────────────────────────

    # Cast filingDate string to a real DateType — F.to_date() returns null for any
    # value it can't parse (e.g. empty string or "N/A"), which is safer than raising an exception
    df = df.withColumn(
        "filingDate",
        F.to_date(F.col("filingDate"), "yyyy-MM-dd")  # SEC uses ISO-8601 date format: "2024-10-30"; F = pyspark.sql.functions alias
    )

    # Count rows where the date cast produced null — these are data-quality issues that
    # should be visible in the logs, not silently swallowed
    bad_date_count: int = df.filter(F.col("filingDate").isNull()).count()
    if bad_date_count > 0:
        print(f"[warn] {bad_date_count} row(s) had unparseable filingDate values and will be removed.")

    # Drop null-date rows — they can't be used in any time-series analysis downstream
    df = df.filter(F.col("filingDate").isNotNull())

    # Extract the calendar year from the date so queries like "all 2023 10-Ks" don't need
    # date arithmetic — a plain integer is faster to filter and easier to read in results
    df = df.withColumn("filing_year", F.year(F.col("filingDate")))

    # ── WHY create_map() INSTEAD OF A PYTHON DICTIONARY LOOKUP ──────────────
    # A Python lookup (called a UDF) is like pulling each item off Spark's assembly
    # line one at a time, handing it to a Python specialist to inspect, then putting
    # it back — repeated for every single row. That's painfully slow at scale.
    # create_map() keeps everything on the assembly line without ever stopping it:
    # Spark handles the lookup internally and can optimize it automatically.
    # Rule of thumb: always keep transformations inside Spark's engine when possible.
    # ────────────────────────────────────────────────────────────────────────────────────────

    # ── WHAT THE LIST COMPREHENSION IS DOING ────────────────────────────────
    # F.create_map() expects a single flat list of alternating keys and values:
    #   [key1, val1, key2, val2, key3, val3, ...]
    #
    # FORM_CATEGORIES.items() produces pairs like ("10-K", "Annual Report").
    # The comprehension loops over each pair and then over each item inside
    # that pair, building the flat list F.create_map() needs.
    #
    # Written out as a plain multi-line loop, it would look like this:
    #
    #   items_list = []
    #   for pair in FORM_CATEGORIES.items():   # pair = ("10-K", "Annual Report")
    #       for item in pair:                  # first item = "10-K", then item = "Annual Report"
    #           items_list.append(F.lit(item)) # F.lit() wraps a Python value so Spark can use it
    #
    # Result: [F.lit("10-K"), F.lit("Annual Report"), F.lit("10-Q"), F.lit("Quarterly Report"), ...]
    # F.create_map() then reads that list as key->value pairs:
    #   "10-K" -> "Annual Report", "10-Q" -> "Quarterly Report", etc.
    # ─────────────────────────────────────────────────────────────────────────

    # Build a Spark-native key->value map from FORM_CATEGORIES so no UDF is needed
    form_map: Column = F.create_map([F.lit(item) for pair in FORM_CATEGORIES.items() for item in pair])

    # ── WHAT IS coalesce()? ──────────────────────────────────────────────────
    # F.coalesce() takes two or more expressions and returns the first one that
    # is NOT null — think of it as a "use this, or fall back to that" instruction.
    #
    # Here it works in two steps:
    #   1. form_map[F.col("form")] — look up the raw SEC code (e.g. "10-K") in
    #      the map built above. If the code exists, return the label ("Annual Report").
    #      If the code is NOT in the map, the lookup returns null.
    #   2. F.lit("Other") — the fallback. If step 1 returned null (code not found),
    #      coalesce returns "Other" instead.
    #
    # End result: known codes get a plain-English label; anything unrecognised
    # gets "Other" rather than a blank/null that would break downstream filters.
    # ─────────────────────────────────────────────────────────────────────────

    # Look up each form code in the map; fall back to "Other" if the code isn't listed
    df = df.withColumn(
        "form_category",
        F.coalesce(form_map[F.col("form")], F.lit("Other"))
    )

    return df


# ── PART 3: RUN ──────────────────────────────────────────────────────────────

filings_clean: Optional[DataFrame] = clean_filings(filings_raw)  # Apply all cleaning and type-casting to the raw DataFrame

if filings_clean is None:
    # ── WHAT DOES "filings_raw is None" MEAN? ───────────────────────────────
    # filings_raw is the Spark DataFrame created at the end of Step 2.
    # It is None when Step 2 produced zero rows — meaning no filing data
    # was ever loaded into Spark.  This can happen for several reasons:
    #
    #   - Step 2 was never run (or was run out of order).
    #   - Every API call to the SEC EDGAR server failed or returned empty data.
    #   - A network or connectivity issue blocked the requests.
    #   - The User-Agent header in HEADERS was incorrect, causing the SEC to
    #     reject all requests with a 403 Forbidden error.
    #
    # HOW TO FIX IT:
    #   1. Scroll up and read any [warn] or [error] lines printed by Step 2.
    #      They usually identify the exact problem (e.g. timeout, 403, empty result).
    #   2. Re-run the Step 2 cell.  A transient network glitch is the most common cause.
    #   3. If the problem persists, verify:
    #        - Your internet connection is active.
    #        - HEADERS["User-Agent"] contains a valid email address.
    #        - QUERY_MODE is set to "SPECIFIC", "FIRST_500", or "ALL" (no typos).
    #        - At least one valid CIK is listed in CIKS_TO_QUERY (for SPECIFIC mode).
    # ─────────────────────────────────────────────────────────────────────────
    print("[error] filings_raw is None — no filing data was loaded in Step 2.")
    print()
    print("  What happened:  Step 2 returned zero rows, so there is nothing to clean here.")
    print("  How to fix it:  Re-run the Step 2 cell and check any [warn]/[error] messages above.")
    print("  Common causes:  network issue, incorrect User-Agent in HEADERS, or Step 2 was skipped.")
else:
    total: int                  = filings_clean.count()
    earliest: Optional[date]    = filings_clean.agg(F.min("filingDate")).collect()[0][0]
    latest: Optional[date]      = filings_clean.agg(F.max("filingDate")).collect()[0][0]

    print(f"Clean filings: {total:,} rows  |  date range: {earliest} -> {latest}")
    print()
    filings_clean.printSchema()  # Confirm each column's confirmed data type after casting
    print()
    filings_clean  # Display a row preview (enabled by eagerEval in Step 1)

## Step 4: Clean & Deduplicate

In [ ]:
# ════════════════════════════════════════════════════════════════
# STEP 4: CLEAN & DEDUPLICATE
# ════════════════════════════════════════════════════════════════
# Goal: Remove rows that would corrupt any downstream analysis —
# specifically rows missing critical fields (nulls) and rows that
# appear more than once in the dataset (duplicates).
#
# This step runs after Step 3 because type-casting must happen
# first: a null filingDate only shows up as null (not as an empty
# string) after F.to_date() has already run.
#
# Deduplicate means to remove duplicate entries from a dataset,
# keeping exactly 1 copy of each unique record.
#
# filings_clean -> filings_deduped (the output used in Step 5+)
# ════════════════════════════════════════════════════════════════


# ── PART 1: CONFIGURATION ────────────────────────────────────────────────────

# Columns that must have a value for a row to be meaningful — a filing
# with no company ID, no date, or no form type can't be used for anything.
REQUIRED_COLUMNS: list[str] = ["cik", "filingDate", "form", "accessionNumber"]

# ── WHY accessionNumber IS THE DEDUP KEY ─────────────────────────────────────
# The SEC EDGAR API assigns every filing a globally unique accessionNumber
# the moment it is submitted — no two filings ever share one.
# Source: https://www.sec.gov/search-filings/edgar-search-assistance/accessing-edgar-data
#
# Deduplicating on this single field is therefore both necessary and sufficient:
# if two rows share the same accessionNumber they are identical records,
# regardless of what the other columns say.  Using all columns at once
# (dropDuplicates() with no arguments) would miss near-duplicates where one row
# has a slightly different value due to a data-entry error on the SEC's end.
# ─────────────────────────────────────────────────────────────────────────────
DEDUP_KEY: list[str] = ["accessionNumber"]  # The SEC's globally unique filing ID — safest dedup key


# ── PART 2: FUNCTIONS ────────────────────────────────────────────────────────

def drop_null_rows(df: Optional[DataFrame], required_cols: list[str]) -> Optional[DataFrame]:
    """
    Removes any row where at least one required column is null.

    Logs exactly how many rows were dropped per column so data-quality
    problems are visible in the output rather than silently discarded.

    Accepts Optional[DataFrame] so the None propagated from upstream steps is
    handled here rather than crashing at the call site. Returns None when df is None.
    """
    if df is None:
        return None  # Nothing to clean — propagate None so the caller handles it

    before: int = df.count()  # Snapshot row count before any filtering starts

    # Check each required column individually so the log shows which field caused removals —
    # a single combined filter would only report a total, hiding which column is the real problem.
    for col in required_cols:
        null_count: int = df.filter(F.col(col).isNull()).count()
        if null_count > 0:
            print(f"[warn] Dropping {null_count:,} row(s) where '{col}' is null.")
        df = df.filter(F.col(col).isNotNull())  # Keep only rows where this column has a value

    after: int   = df.count()
    removed: int = before - after

    if removed == 0:
        print(f"[info] Null check passed — no rows removed (all {before:,} rows have required fields).")
    else:
        print(f"[info] Null removal: {before:,} rows in -> {after:,} rows out ({removed:,} removed).")

    return df


def drop_duplicate_rows(df: Optional[DataFrame], dedup_key: list[str]) -> Optional[DataFrame]:
    """
    Removes rows that share the same value in dedup_key columns.

    When duplicates exist, Spark keeps one row arbitrarily (there is no
    guarantee about which copy is retained).  For SEC EDGAR data this is fine
    because identical accessionNumbers mean the rows are byte-for-byte
    the same record — it doesn't matter which copy survives.

    Logs how many duplicate rows were dropped so nothing is lost silently.
    Accepts Optional[DataFrame] so the None propagated from drop_null_rows is
    handled here rather than crashing at the call site. Returns None when df is None.
    """
    if df is None:
        return None  # Propagate None — no data to deduplicate

    before: int = df.count()  # Count before dedup so the log can show what was removed

    # dropDuplicates() with a column list keeps one row per unique combination of those columns
    # and discards the rest — Spark handles the partitioned shuffle internally.
    df = df.dropDuplicates(dedup_key)

    after: int   = df.count()
    removed: int = before - after

    if removed == 0:
        print(f"[info] Duplicate check passed — no duplicates found ({before:,} rows, key: {dedup_key}).")
    else:
        print(f"[info] Dedup: {before:,} rows in -> {after:,} rows out ({removed:,} duplicate(s) removed, key: {dedup_key}).")

    return df


def clean_and_deduplicate_filings(df: Optional[DataFrame]) -> Optional[DataFrame]:
    """
    Runs both cleaning passes in sequence and returns the final DataFrame:
      1. Drop rows missing any required field (null removal)
      2. Drop rows sharing the same accessionNumber (duplicate removal)

    Named "clean_and_deduplicate" rather than just "deduplicate" because this
    function handles two distinct problems — missing values AND exact duplicates.
    Keeping these as two separate passes (rather than one combined filter)
    makes the log output easier to read and isolates each type of problem.

    The Optional[DataFrame] parameter type signals that None is a valid input
    (meaning no data arrived from an earlier step), not a bug to crash on.

    Returns filings_deduped ready for analysis in Step 5+, or None if df is None.
    """
    if df is None:
        return None

    print("── Null Removal ─────────────────────────────────────────────")
    df = drop_null_rows(df, REQUIRED_COLUMNS)  # Remove rows where any critical field is missing

    print("── Duplicate Removal ────────────────────────────────────────")
    df = drop_duplicate_rows(df, DEDUP_KEY)    # Remove rows that are exact copies by accessionNumber

    return df


# ── PART 3: RUN ──────────────────────────────────────────────────────────────

filings_deduped: Optional[DataFrame] = clean_and_deduplicate_filings(filings_clean)  # Apply null and duplicate removal to Step 3 output

if filings_deduped is None:
    print("[error] filings_clean is None — Step 3 produced no data.")
    print()
    print("  What happened:  This step depends on filings_clean from Step 3.")
    print("  How to fix it:  Run Steps 1, 2, and 3 in order before running this cell.")
else:
    total: int = filings_deduped.count()
    print()
    print(f"Final dataset: {total:,} rows ready for analysis.")
    print()
    filings_deduped  # Display a row preview (enabled by eagerEval in Step 1)

## Step 5: Transform (Date Columns & Derived Fields)

In [ ]:
# ════════════════════════════════════════════════════════════════
# STEP 5: TRANSFORM — DATE COLUMNS & DERIVED FIELDS
# ════════════════════════════════════════════════════════════════
# Goal: Enrich filings_deduped with additional columns that make
# downstream analysis easier to write and faster to run.
#
# Step 3 gave every row a real DateType and a filing_year.
# Step 5 goes further, adding:
#   - filing_quarter  — which quarter of the year (Q1–Q4)
#   - filing_month    — integer month (1–12) for month-level grouping
#   - days_since_filing — how many calendar days ago the filing was submitted
#   - is_recent       — boolean flag: filed within the last 2 years
#   - report_type     — broader category grouping related form types together
#
# These columns exist purely for convenience — everything here could be
# computed on the fly in a query, but pre-computing them once means every
# downstream step can filter and group without repeating date arithmetic.
#
# Input:  filings_deduped   (from Step 4)
# Output: filings_transformed (used in Step 6+)
# ════════════════════════════════════════════════════════════════


# ── PART 1: CONFIGURATION ────────────────────────────────────────────────────

# How many years back a filing must be to qualify as "recent" —
# changing this one value updates the is_recent flag for every row.
RECENT_YEARS_THRESHOLD: int = 2

# ── WHY GROUP FORMS INTO report_type? ────────────────────────────────────────
# SEC form codes are precise but numerous — there are hundreds of variants.
# Grouping them into four broad buckets (Periodic, Event-Driven, Insider, Other)
# lets downstream queries answer questions like "how many regulatory disclosures
# did Apple file in 2023?" without needing to know every form code variant.
# The map is defined here (not buried in a function) so it's easy to extend.
# ─────────────────────────────────────────────────────────────────────────────
REPORT_TYPE_MAP: dict[str, str] = {
    "10-K":    "Periodic",       # Annual report — filed once per fiscal year
    "10-K/A":  "Periodic",       # Amended annual report
    "10-Q":    "Periodic",       # Quarterly report — filed three times per fiscal year
    "10-Q/A":  "Periodic",       # Amended quarterly report
    "8-K":     "Event-Driven",   # Material event disclosure — filed within 4 business days of the event
    "8-K/A":   "Event-Driven",   # Amendment to a previous 8-K
    "4":       "Insider",        # Insider trade report — filed within 2 days of a director/officer transaction
    "4/A":     "Insider",        # Amendment to a previous Form 4
    "DEF 14A": "Proxy",          # Definitive proxy statement — notice of shareholder vote (e.g. board elections)
    "S-1":     "Registration",   # IPO registration statement — filed before a company goes public
    "S-1/A":   "Registration",   # Amendment to a previous S-1
}


# ── PART 2: FUNCTIONS ────────────────────────────────────────────────────────

def add_date_columns(df: Optional[DataFrame]) -> Optional[DataFrame]:
    """
    Adds filing_month and filing_quarter to the DataFrame.

    filing_month  — integer 1–12 extracted directly from filingDate.
    filing_quarter — string "Q1"/"Q2"/"Q3"/"Q4" derived from the month using
                     F.ceil(month / 3), which maps months to quarters:
                       month 1–3  -> ceil(1/3)=1 -> "Q1"
                       month 4–6  -> ceil(4/3)=2 -> "Q2"  (ceil rounds up)
                       month 7–9  -> ceil(7/3)=3 -> "Q3"
                       month 10–12 -> ceil(10/3)=4 -> "Q4"
                     The "Q" prefix is added with F.concat so the result
                     reads "Q1" rather than just "1".

    Returns None unchanged so the None propagates cleanly through the pipeline.
    """
    if df is None:
        return None

    # Extract month as an integer so quarter math works without string parsing
    df = df.withColumn("filing_month", F.month(F.col("filingDate")))

    # ── HOW F.ceil() MAPS MONTHS TO QUARTERS ────────────────────────────────
    # Dividing the month number by 3 and rounding up gives the quarter index:
    #   Jan=1  -> ceil(1/3)  = ceil(0.33) = 1  -> Q1
    #   Mar=3  -> ceil(3/3)  = ceil(1.00) = 1  -> Q1
    #   Apr=4  -> ceil(4/3)  = ceil(1.33) = 2  -> Q2
    #   Dec=12 -> ceil(12/3) = ceil(4.00) = 4  -> Q4
    # F.cast("int") keeps the result as a plain integer so concat() can
    # format it as a string.  F.lit(3) wraps the constant so Spark
    # can divide a column by it — you can't mix a Column and a bare Python int.
    # ─────────────────────────────────────────────────────────────────────────

    # Derive quarter number (1–4) from month, then prefix it with "Q" for readability
    df = df.withColumn(
        "filing_quarter",
        F.concat(
            F.lit("Q"),
            F.ceil(F.col("filing_month") / F.lit(3)).cast("int")  # ceil(month/3) maps month -> quarter index
        )
    )

    return df


def add_recency_columns(df: Optional[DataFrame], recent_years: int) -> Optional[DataFrame]:
    """
    Adds days_since_filing and is_recent to the DataFrame.

    days_since_filing — how many calendar days have passed since the filing date,
                        calculated relative to today's date at run time.
                        Useful for sorting by recency or computing filing frequency.

    is_recent         — boolean True/False: True when days_since_filing falls within
                        recent_years * 365 days.  Using days instead of year subtraction
                        avoids edge cases around leap years and partial calendar years.

    F.current_date() is evaluated at query execution time by Spark, not at the moment
    this Python function is called — so the result is always accurate regardless of
    when the notebook was last run.

    Returns None unchanged so the None propagates cleanly through the pipeline.
    """
    if df is None:
        return None

    # F.datediff(end, start) returns the number of days between two dates as an integer —
    # positive when end is after start, which is always true here since filingDate is in the past.
    df = df.withColumn(
        "days_since_filing",
        F.datediff(F.current_date(), F.col("filingDate"))  # Days elapsed since the filing was submitted
    )

    # Convert years to a day threshold once so the formula is readable at a glance.
    # Using days (not year subtraction) avoids off-by-one issues around partial calendar years.
    recent_day_threshold: int = recent_years * 365
    df = df.withColumn(
        "is_recent",
        # Here, <= means "less than or equal to" — Spark evaluates this per row and produces a boolean:
        # True if the filing's age is less than the threshold window (Ex: 400 days <= 730 days), False if older.
        F.col("days_since_filing") <= F.lit(recent_day_threshold)
    )

    return df


def add_report_type(df: Optional[DataFrame]) -> Optional[DataFrame]:
    """
    Adds a report_type column that maps raw SEC form codes to a broader category.

    The mapping is driven by REPORT_TYPE_MAP (defined in Part 1).
    Unknown form codes fall back to "Other" via F.coalesce() rather than
    leaving a null that would break GROUP BY queries downstream.

    This uses the same Spark-native create_map() pattern as Step 3's form_category,
    avoiding a Python UDF so the transformation stays inside Spark's engine.

    Returns None unchanged so the None propagates cleanly through the pipeline.
    """
    if df is None:
        return None

    # Build a Spark-native lookup map from REPORT_TYPE_MAP — same pattern as Step 3's form_map.
    # F.create_map() takes a flat alternating list [key1, val1, key2, val2, ...];
    # the list comprehension unpacks REPORT_TYPE_MAP.items() into that format.
    report_map: Column = F.create_map(
        [F.lit(item) for pair in REPORT_TYPE_MAP.items() for item in pair]
    )

    # Look up the form code; fall back to "Other" for any code not in the map
    df = df.withColumn(
        "report_type",
        F.coalesce(report_map[F.col("form")], F.lit("Other"))  # "Other" prevents nulls for unrecognised codes
    )

    return df


def transform_filings(df: Optional[DataFrame]) -> Optional[DataFrame]:
    """
    Applies all Step 5 transformations in sequence and returns the enriched DataFrame.

    Calling order matters:
      1. add_date_columns   — must run before add_recency_columns because
                              days_since_filing depends on filingDate being a DateType,
                              which was set in Step 3 (already done).
      2. add_recency_columns — adds days_since_filing and is_recent.
      3. add_report_type    — independent of the date columns; order doesn't matter here.

    Returns filings_transformed ready for Step 6 (broadcast join), or None if df is None.
    """
    if df is None:
        return None

    df = add_date_columns(df)                              # Add filing_month and filing_quarter
    df = add_recency_columns(df, RECENT_YEARS_THRESHOLD)   # Add days_since_filing and is_recent
    df = add_report_type(df)                               # Add report_type broad category

    return df


# ── PART 3: RUN ──────────────────────────────────────────────────────────────

filings_transformed: Optional[DataFrame] = transform_filings(filings_deduped)  # Enrich the clean dataset with derived columns

if filings_transformed is None:
    print("[error] filings_deduped is None — no data arrived from Step 4.")
    print()
    print("  What happened:  This step depends on filings_deduped from Step 4.")
    print("  How to fix it:  Run Steps 1 through 4 in order before running this cell.")
else:
    total: int             = filings_transformed.count()
    recent_count: int      = filings_transformed.filter(F.col("is_recent")).count()
    recent_pct: float      = (recent_count / total * 100) if total > 0 else 0.0

    print(f"Transformed dataset: {total:,} rows | {recent_count:,} recent filings ({recent_pct:.1f}% within last {RECENT_YEARS_THRESHOLD} years)")
    print()
    filings_transformed.printSchema()  # Confirm the new columns appear with the correct types
    print()
    filings_transformed  # Display a row preview (enabled by eagerEval in Step 1)